# Bước 2: Tiền xử lý Dữ liệu (Data Preprocessing)

Dựa trên kết quả từ bước EDA, nhóm sẽ thực hiện các công việc sau:
1. **Loại bỏ cột không cần thiết**: `CLIENTNUM` và các cột Naive Bayes.
2. **Xử lý giá trị 'Unknown'**: Thay thế bằng giá trị xuất hiện nhiều nhất (Mode) của cột đó.
3. **Mã hóa (Encoding)**: Chuyển đổi các biến phân loại thành dạng số để mô hình có thể hiểu được.
4. **Lưu trữ dữ liệu sạch**: Để phục vụ cho các bước huấn luyện mô hình sau này.

In [1]:
import pandas as pd
import numpy as np

# 1. Load dữ liệu
df = pd.read_csv('../data/BankChurners.csv')

# Loại bỏ các cột không cần thiết
cols_to_drop = [col for col in df.columns if col.startswith('Naive_Bayes')] + ['CLIENTNUM']
df = df.drop(columns=cols_to_drop)

print(f"Cấu trúc dữ liệu sau khi load: {df.shape}")
df.head()

## 2. Xử lý giá trị 'Unknown'

Trong tập dữ liệu, có một số cột chứa giá trị 'Unknown'. Chúng ta sẽ thay bằng giá trị phổ biến nhất (mode).

In [2]:
# Danh các cột có giá trị 'Unknown'
unknown_cols = ['Education_Level', 'Marital_Status', 'Income_Category']

for col in unknown_cols:
    # Tính mode (giá trị phổ biến nhất) không tính 'Unknown'
    mode_val = df[df[col] != 'Unknown'][col].mode()[0]
    # Thay thế 'Unknown' bằng mode_val
    df[col] = df[col].replace('Unknown', mode_val)
    print(f"Column '{col}': Replaced 'Unknown' with '{mode_val}'")

print("\nKiểm tra lại xem còn 'Unknown' không:")
print((df == 'Unknown').sum().sum())

## 3. Mã hóa (Encoding)

Chúng ta sẽ thực hiện:
- **Label Encoding** cho biến mục tiêu và các biến có tính thứ tự (Ordinal).
- **One-Hot Encoding** cho các biến định danh (Nominal) không có tính thứ tự rõ rệt.

In [3]:
# 3.1. Mã hóa biến mục tiêu Attrition_Flag
df['Attrition_Flag'] = df['Attrition_Flag'].map({'Existing Customer': 0, 'Attrited Customer': 1})

# 3.2. Mã hóa Gender (M: 1, F: 0)
df['Gender'] = df['Gender'].map({'M': 1, 'F': 0})

# 3.3. Mã hóa Ordinal Features (Có tính thứ bậc)

# Education_Level
edu_map = {
    'Uneducated': 0, 
    'High School': 1, 
    'College': 2, 
    'Graduate': 3, 
    'Post-Graduate': 4, 
    'Doctorate': 5
}
df['Education_Level'] = df['Education_Level'].map(edu_map)

# Income_Category
income_map = {
    'Less than $40K': 0, 
    '$40K - $60K': 1, 
    '$60K - $80K': 2, 
    '$80K - $120K': 3, 
    '$120K +': 4
}
df['Income_Category'] = df['Income_Category'].map(income_map)

# Card_Category
card_map = {
    'Blue': 0, 
    'Silver': 1, 
    'Gold': 2, 
    'Platinum': 3
}
df['Card_Category'] = df['Card_Category'].map(card_map)

# 3.4. Mã hóa Nominal Features (One-Hot Encoding)
# Cột Marital_Status
df = pd.get_dummies(df, columns=['Marital_Status'], prefix='Marital')

print("Dữ liệu sau khi mã hóa:")
df.head()

## 4. Lưu dữ liệu sạch

Lưu file csv mới để sử dụng cho các bước tiếp theo.

In [4]:
# Lưu dữ liệu sạch
df.to_csv('../data/BankChurners_cleaned.csv', index=False)
print("Đã lưu dữ liệu sạch vào data/BankChurners_cleaned.csv")